# Identificação do projeto

**Título do projeto:** Análise operacional de entregas de comida: tempo, custo e satisfação por pedido  
**Integrantes do grupo:** Carlos Eduardo Wille Martins, Davi Batista Amaral, Enzo Bortolozzo Borges e Henrique Fugikawa Abe  
**Turma:** Sistemas de Informação • GCL1 • 6º período (Profa. Marina de Lara)  
**Descrição breve do problema ou contexto:** Entender como distância, tráfego, clima e perfil do pedido se relacionam com o tempo de entrega, a receita, o lucro e a satisfação em uma plataforma de entrega de comida (base sintética de 150.000 pedidos).

Preencha este notebook ao longo dos dois RAs. Registre decisões e interpretações em poucas frases e acrescente as células de código necessárias em cada etapa.

---

# RA1 — Data Ingestion, Cleaning and Initial Analysis

Nesta etapa, carregue, prepare e explore os dados do projeto.

## 1. Contexto e objetivo

Apresente brevemente o problema investigado e o que o grupo pretende descobrir com os dados.

**Resposta do grupo:**

A entrega de comida por aplicativo depende de uma operação logística em que o tempo de entrega influencia a satisfação, os custos e a retenção de clientes. O problema investigado é entender quais características do pedido e da operação (distância, tráfego, clima, tipo de restaurante, culinária, cidade e perfil do cliente) estão associadas a entregas mais demoradas, a menor satisfação e a diferenças de receita e lucro.

Perguntas analíticas que orientam o projeto:

- **P1:** Como o tempo de entrega varia conforme a distância, o nível de tráfego e a condição climática?
- **P2:** Como a receita e o lucro por pedido diferem entre tipos de restaurante, tipos de culinária e cidades?
- **P3:** O tempo de entrega está associado à avaliação do cliente, às reclamações e aos reembolsos?
- **P4:** Como o volume de pedidos, a receita e o lucro variam entre os meses, os trimestres e as cidades?
- **P5:** Quais perfis de cliente (tipo e idade) apresentam maior risco de churn e maior valor de pedido?

Neste RA1, o foco é obter e preparar a base. Análises, agregações e conclusões ficam para as próximas etapas.

## 2. Fonte e ingestão dos dados

- Identifique a fonte dos dados, incluindo link ou referência de acesso.
- Descreva brevemente o dataset e o que seus registros representam.
- Explique como os dados foram carregados e implemente a ingestão.

**Resposta do grupo:**

- **Fonte:** Food Delivery Operations Analytics Dataset (autor roshaaann30, Kaggle): https://www.kaggle.com/datasets/roshaaann30/food-delivery-operations-analytics-dataset (acesso em 15/10/2026).
- **Licença:** CC BY-NC-SA 4.0.
- **Descrição:** segundo o Kaggle, é uma base **sintética** que simula plataformas reais de entrega. Cada linha é um **pedido**; são 150.000 registros e 21 colunas (operação, pedido, cliente e contexto: cidade, mês e trimestre, sem ano).
- **Ingestão:** o CSV é baixado do Kaggle por código com a biblioteca `kagglehub` e carregado em um DataFrame pandas (`df`). Uma cópia intacta (`df_raw`) é guardada para a limpeza e a comparação antes/depois.

In [1]:
# Instala o kagglehub com suporte a pandas (só precisa na primeira vez no Colab)
!pip install -q "kagglehub[pandas-datasets]"

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

pd.set_option("display.max_columns", None)
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# Ingestão: carrega o dataset diretamente do Kaggle para um DataFrame pandas
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "roshaaann30/food-delivery-operations-analytics-dataset",
    "Food_Delivery_Operations_Analytics.csv"
)

# Cópia intacta dos dados brutos (para a comparação antes/depois)
df_raw = df.copy()
df_raw.to_csv("data/raw/Food_Delivery_Operations_Analytics.csv", index=False)

print(df.head())
print("\nRegistros:", df.shape[0], "| Colunas:", df.shape[1])
print("Ambiente:", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "| kagglehub", kagglehub.__version__)

100%|██████████| 16.2M/16.2M [00:02<00:00, 7.37MB/s]


     Order_ID  Customer_Age Customer_Type Restaurant_Type Cuisine_Type  \
0  ORD0000001            28       Premium   Cloud Kitchen      Italian   
1  ORD0000002            46           New            Cafe      Italian   
2  ORD0000003            37           New            Cafe      Mexican   
3  ORD0000004            52       Premium       Fast Food      Italian   
4  ORD0000005            40           New   Cloud Kitchen      Mexican   

   Delivery_Distance_KM  Delivery_Time_Min  Order_Value_USD  Item_Count  \
0                  7.13                104           192.24           9   
1                 15.64                106           276.43           6   
2                 11.79                 56           251.85           9   
3                 18.36                 17           285.04          13   
4                 20.33                 83            36.21          10   

  Weather_Condition Traffic_Level  Customer_Rating  Complaint_Flag  \
0            Cloudy           Low 

## 3. Exploração inicial dos dados

Investigue a quantidade de registros e colunas, os tipos de dados, os valores ausentes e outras características relevantes da base.

In [3]:
# Estrutura, tipos e resumo estatístico
df.info()
display(df.describe(include="all").T)

cat_cols = ["Customer_Type", "Restaurant_Type", "Cuisine_Type",
            "Weather_Condition", "Traffic_Level", "City"]
num_cols = ["Customer_Age", "Delivery_Distance_KM", "Delivery_Time_Min", "Order_Value_USD",
            "Item_Count", "Customer_Rating", "Revenue_USD", "Profit_USD",
            "Demand_Score", "Churn_Risk"]

for c in cat_cols:
    print(f"{c}:", df[c].unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Order_ID              150000 non-null  object 
 1   Customer_Age          150000 non-null  int64  
 2   Customer_Type         150000 non-null  object 
 3   Restaurant_Type       150000 non-null  object 
 4   Cuisine_Type          150000 non-null  object 
 5   Delivery_Distance_KM  150000 non-null  float64
 6   Delivery_Time_Min     150000 non-null  int64  
 7   Order_Value_USD       150000 non-null  float64
 8   Item_Count            150000 non-null  int64  
 9   Weather_Condition     150000 non-null  object 
 10  Traffic_Level         150000 non-null  object 
 11  Customer_Rating       150000 non-null  float64
 12  Complaint_Flag        150000 non-null  int64  
 13  Refund_Flag           150000 non-null  int64  
 14  Revenue_USD           150000 non-null  float64
 15  

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Order_ID,150000,150000,ORD0149984,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer_Age,150000.0,NaN,NaN,NaN,46.047953,16.45781,18.0,32.0,46.0,60.0,74.0
Customer_Type,150000,3,New,50185,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Restaurant_Type,150000,4,Cafe,37743,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Cuisine_Type,150000,4,Indian,37635,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Delivery_Distance_KM,150000.0,NaN,NaN,NaN,12.761809,7.082171,0.5,6.63,12.76,18.91,25.0
Delivery_Time_Min,150000.0,NaN,NaN,NaN,64.52824,31.742264,10.0,37.0,65.0,92.0,119.0
Order_Value_USD,150000.0,NaN,NaN,NaN,152.888794,85.088266,5.0,79.24,153.125,226.69,300.0
Item_Count,150000.0,NaN,NaN,NaN,7.5081,4.031177,1.0,4.0,8.0,11.0,14.0
Weather_Condition,150000,4,Sunny,37610,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Customer_Type: ['Premium' 'New' 'Returning']
Restaurant_Type: ['Cloud Kitchen' 'Cafe' 'Fast Food' 'Restaurant']
Cuisine_Type: ['Italian' 'Mexican' 'Chinese' 'Indian']
Weather_Condition: ['Cloudy' 'Sunny' 'Rainy' 'Stormy']
Traffic_Level: ['Low' 'High' 'Medium']
City: ['Singapore' 'New York' 'Mumbai' 'London' 'Sydney']


In [4]:
# Q1: valores ausentes
nulos = df.isna().sum().to_frame("ausentes")
nulos["%"] = (nulos["ausentes"] / len(df) * 100).round(2)
display(nulos)

# Q2: linhas duplicadas
print("Linhas duplicadas:", df.duplicated().sum())

# Q3: Order_ID (unicidade e formato)
print("Order_ID repetidos:", df["Order_ID"].duplicated().sum())
fora_formato = ~df["Order_ID"].astype(str).str.match(r"^ORD\d{7}$")
print("Order_ID fora do formato ORD + 7 dígitos:", fora_formato.sum())

# Q4: tipos de dados
print(df.dtypes)
for c in num_cols:
    if not pd.api.types.is_numeric_dtype(df[c]):
        print("Tipo incorreto:", c, df[c].dtype)

,ausentes,%
Order_ID,0,0.0
Customer_Age,0,0.0
Customer_Type,0,0.0
Restaurant_Type,0,0.0
Cuisine_Type,0,0.0
Delivery_Distance_KM,0,0.0
Delivery_Time_Min,0,0.0
Order_Value_USD,0,0.0
Item_Count,0,0.0
Weather_Condition,0,0.0


Linhas duplicadas: 0
Order_ID repetidos: 0
Order_ID fora do formato ORD + 7 dígitos: 0
Order_ID                 object
Customer_Age              int64
Customer_Type            object
Restaurant_Type          object
Cuisine_Type             object
Delivery_Distance_KM    float64
Delivery_Time_Min         int64
Order_Value_USD         float64
Item_Count                int64
Weather_Condition        object
Traffic_Level            object
Customer_Rating         float64
Complaint_Flag            int64
Refund_Flag               int64
Revenue_USD             float64
Profit_USD              float64
City                     object
Month                     int64
Quarter                   int64
Demand_Score            float64
Churn_Risk              float64
dtype: object


In [5]:
# Q5: idade plausível
print(df["Customer_Age"].describe())
print("Fora de 10–100 anos:", (~df["Customer_Age"].between(10, 100)).sum())

# Q6: valores menores ou iguais a zero
for c in ["Delivery_Distance_KM", "Delivery_Time_Min", "Order_Value_USD", "Item_Count"]:
    print(f"{c}: {(df[c] <= 0).sum()} valores <= 0")

# Q7: categorias inconsistentes
for c in cat_cols:
    print(f"\n{c}")
    print(df[c].value_counts(dropna=False))

count    150000.000000
mean         46.047953
std          16.457810
min          18.000000
25%          32.000000
50%          46.000000
75%          60.000000
max          74.000000
Name: Customer_Age, dtype: float64
Fora de 10–100 anos: 0
Delivery_Distance_KM: 0 valores <= 0
Delivery_Time_Min: 0 valores <= 0
Order_Value_USD: 0 valores <= 0
Item_Count: 0 valores <= 0

Customer_Type
Customer_Type
New          50185
Returning    49912
Premium      49903
Name: count, dtype: int64

Restaurant_Type
Restaurant_Type
Cafe             37743
Fast Food        37528
Cloud Kitchen    37387
Restaurant       37342
Name: count, dtype: int64

Cuisine_Type
Cuisine_Type
Indian     37635
Italian    37542
Mexican    37420
Chinese    37403
Name: count, dtype: int64

Weather_Condition
Weather_Condition
Sunny     37610
Cloudy    37497
Stormy    37447
Rainy     37446
Name: count, dtype: int64

Traffic_Level
Traffic_Level
Low       50105
Medium    49997
High      49898
Name: count, dtype: int64

City
City
Sin

In [6]:
# Q8: possíveis outliers (regra do IQR, só diagnóstico)
def contar_outliers_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()

res = pd.DataFrame({
    "outliers": {c: contar_outliers_iqr(df[c]) for c in
                 ["Delivery_Distance_KM", "Delivery_Time_Min", "Order_Value_USD", "Item_Count"]}
})
res["%"] = (res["outliers"] / len(df) * 100).round(2)
display(res)

# Q9: coerência entre tempo e distância (velocidade implícita, km/h)
vel = df["Delivery_Distance_KM"] / (df["Delivery_Time_Min"] / 60)
print(vel.describe())
print("Velocidade > 80 km/h:", (vel > 80).sum())
print("Velocidade < 3 km/h:", (vel < 3).sum())
print(df[["Delivery_Time_Min", "Delivery_Distance_KM"]].corr())

,outliers,%
Delivery_Distance_KM,0,0.0
Delivery_Time_Min,0,0.0
Order_Value_USD,0,0.0
Item_Count,0,0.0


count    150000.000000
mean         17.580147
std          19.024186
min           0.252101
25%           6.190909
50%          11.876331
75%          20.856570
max         149.880000
dtype: float64
Velocidade > 80 km/h: 3159
Velocidade < 3 km/h: 16653
                      Delivery_Time_Min  Delivery_Distance_KM
Delivery_Time_Min              1.000000              0.004071
Delivery_Distance_KM           0.004071              1.000000


In [7]:
# Q10: flags apenas 0 e 1
for c in ["Complaint_Flag", "Refund_Flag"]:
    print(c, sorted(df[c].unique()))

# Q11: mês entre 1 e 12 e trimestre coerente
print("Month fora de 1-12:", (~df["Month"].between(1, 12)).sum())
print("Quarter incoerente com Month:", (df["Quarter"] != (df["Month"] - 1) // 3 + 1).sum())

# Q12: faixas de Customer_Rating, Demand_Score e Churn_Risk
print(df[["Customer_Rating", "Demand_Score", "Churn_Risk"]].agg(["min", "max"]))

# Q13: lucro maior que a receita
print("Profit > Revenue:", (df["Profit_USD"] > df["Revenue_USD"]).sum())

Complaint_Flag [np.int64(0), np.int64(1)]
Refund_Flag [np.int64(0), np.int64(1)]
Month fora de 1-12: 0
Quarter incoerente com Month: 0
     Customer_Rating  Demand_Score  Churn_Risk
min              1.0           0.0         0.0
max              5.0         100.0       100.0
Profit > Revenue: 26255


**Resposta do grupo:**

- **Tamanho:** 150.000 registros e 21 colunas, uma linha por pedido. Tipos corretos (numéricas em int64/float64 e categóricas em texto).
- **Qualidade:** sem valores ausentes (Q1), sem duplicados (Q2), `Order_ID` único e no formato ORD + 7 dígitos (Q3), idade de 18 a 74 anos (Q5), nenhum valor menor ou igual a zero (Q6), categorias consistentes (Q7), nenhum outlier pelo IQR (Q8), flags só com 0 e 1 (Q10), mês e trimestre coerentes (Q11) e avaliação de 1 a 5 (Q12).
- **Dois pontos de atenção:** (Q9) a correlação entre tempo e distância é praticamente nula (0,004) e há entregas com velocidade irreal: 3.159 acima de 80 km/h e 16.653 abaixo de 3 km/h; (Q13) 26.255 registros (17,5%) têm lucro maior que a receita.
- **Distribuições:** uniformes e com categorias equilibradas, o que é coerente com uma base sintética.

## 4. Limpeza e preparação dos dados

Realize a limpeza necessária ao projeto. Verifique valores ausentes, duplicados e inconsistências e trate os problemas encontrados, quando necessário.

In [8]:
# Cópia de trabalho e registro de quantos registros cada etapa remove
df_clean = df_raw.copy()
log = {"inicio": len(df_clean)}

# D3: correção de tipos (numéricas conferidas; texto convertido para string)
for c in num_cols:
    df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")
df_clean[cat_cols + ["Order_ID"]] = df_clean[cat_cols + ["Order_ID"]].astype("string")

# D5: padronização preventiva de categorias
for c in cat_cols:
    df_clean[c] = df_clean[c].str.strip().str.title()
for c in cat_cols:
    print(f"{c}:", sorted(df_clean[c].dropna().unique()))

Customer_Type: ['New', 'Premium', 'Returning']
Restaurant_Type: ['Cafe', 'Cloud Kitchen', 'Fast Food', 'Restaurant']
Cuisine_Type: ['Chinese', 'Indian', 'Italian', 'Mexican']
Weather_Condition: ['Cloudy', 'Rainy', 'Stormy', 'Sunny']
Traffic_Level: ['High', 'Low', 'Medium']
City: ['London', 'Mumbai', 'New York', 'Singapore', 'Sydney']


In [9]:
# D1: valores ausentes (remove registros sem campos-chave; imputa o restante)
chaves = ["Order_ID", "Delivery_Distance_KM", "Delivery_Time_Min", "Order_Value_USD"]
antes = len(df_clean)
df_clean = df_clean.dropna(subset=chaves)
log["D1_removidos_chave"] = antes - len(df_clean)
for c in ["Customer_Age", "Item_Count"]:
    df_clean[c] = df_clean[c].fillna(df_clean[c].median())
for c in cat_cols:
    df_clean[c] = df_clean[c].fillna("Unknown")

# D2: duplicados
antes = len(df_clean)
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.drop_duplicates(subset="Order_ID", keep="first")
log["D2_removidos"] = antes - len(df_clean)
print(log)

{'inicio': 150000, 'D1_removidos_chave': 0, 'D2_removidos': 0}


In [10]:
# D4: valores impossíveis
validos = (
    df_clean["Customer_Age"].between(10, 100)
    & (df_clean["Delivery_Distance_KM"] > 0)
    & (df_clean["Delivery_Time_Min"] > 0)
    & (df_clean["Order_Value_USD"] > 0)
    & (df_clean["Item_Count"] > 0)
    & df_clean["Customer_Rating"].between(1, 5)
    & df_clean["Complaint_Flag"].isin([0, 1])
    & df_clean["Refund_Flag"].isin([0, 1])
    & df_clean["Month"].between(1, 12)
    & (df_clean["Quarter"] == (df_clean["Month"] - 1) // 3 + 1)
)
log["D4_removidos"] = int((~validos).sum())
df_clean = df_clean[validos]

# D6: tempo x distância extremos (Q9): registros mantidos
log["D6_removidos"] = 0

# D7: lucro maior que a receita (Q13): registros mantidos
log["D7_removidos"] = 0
print("Profit > Revenue mantidos:", (df_clean["Profit_USD"] > df_clean["Revenue_USD"]).sum())

df_clean = df_clean.reset_index(drop=True)
log["fim"] = len(df_clean)
print(log)

Profit > Revenue mantidos: 26255
{'inicio': 150000, 'D1_removidos_chave': 0, 'D2_removidos': 0, 'D4_removidos': 0, 'D6_removidos': 0, 'D7_removidos': 0, 'fim': 150000}


In [11]:
# Verificações finais do conjunto preparado
print("Tipos:\n", df_clean.dtypes, "\n")
print("Ausentes:", df_clean.isna().sum().sum())
print("Linhas duplicadas:", df_clean.duplicated().sum())
print("Order_ID único:", df_clean["Order_ID"].is_unique)
display(df_clean[num_cols].describe())
for c in cat_cols:
    print(f"{c}:", df_clean[c].value_counts().to_dict())

comparacao = pd.DataFrame({
    "Após ingestão": [len(df_raw), df_raw.shape[1], df_raw.isna().sum().sum(),
                      df_raw.duplicated().sum(), df_raw["Order_ID"].duplicated().sum()],
    "Após preparação": [len(df_clean), df_clean.shape[1], df_clean.isna().sum().sum(),
                        df_clean.duplicated().sum(), df_clean["Order_ID"].duplicated().sum()],
}, index=["Registros", "Atributos", "Valores ausentes", "Linhas duplicadas", "Order_ID repetidos"])
display(comparacao)

Tipos:
 Order_ID                string[python]
Customer_Age                     int64
Customer_Type           string[python]
Restaurant_Type         string[python]
Cuisine_Type            string[python]
Delivery_Distance_KM           float64
Delivery_Time_Min                int64
Order_Value_USD                float64
Item_Count                       int64
Weather_Condition       string[python]
Traffic_Level           string[python]
Customer_Rating                float64
Complaint_Flag                   int64
Refund_Flag                      int64
Revenue_USD                    float64
Profit_USD                     float64
City                    string[python]
Month                            int64
Quarter                          int64
Demand_Score                   float64
Churn_Risk                     float64
dtype: object 

Ausentes: 0
Linhas duplicadas: 0
Order_ID único: True


,Customer_Age,Delivery_Distance_KM,Delivery_Time_Min,Order_Value_USD,Item_Count,Customer_Rating,Revenue_USD,Profit_USD,Demand_Score,Churn_Risk
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000
mean,46.047953,12.761809,64.528240,152.888794,7.508100,2.999477,252.287419,90.032737,50.057309,49.926011
std,16.457810,7.082171,31.742264,85.088266,4.031177,1.156920,142.917771,63.488361,28.899435,28.921656
min,18.000000,0.500000,10.000000,5.000000,1.000000,1.000000,5.000000,-20.000000,0.000000,0.000000
25%,32.000000,6.630000,37.000000,79.240000,4.000000,2.000000,128.430000,35.060000,24.960000,24.880000
50%,46.000000,12.760000,65.000000,153.125000,8.000000,3.000000,251.910000,90.060000,50.060000,49.830000
75%,60.000000,18.910000,92.000000,226.690000,11.000000,4.000000,375.962500,145.160000,75.170000,75.000000
max,74.000000,25.000000,119.000000,300.000000,14.000000,5.000000,500.000000,200.000000,100.000000,100.000000


Customer_Type: {'New': 50185, 'Returning': 49912, 'Premium': 49903}
Restaurant_Type: {'Cafe': 37743, 'Fast Food': 37528, 'Cloud Kitchen': 37387, 'Restaurant': 37342}
Cuisine_Type: {'Indian': 37635, 'Italian': 37542, 'Mexican': 37420, 'Chinese': 37403}
Weather_Condition: {'Sunny': 37610, 'Cloudy': 37497, 'Stormy': 37447, 'Rainy': 37446}
Traffic_Level: {'Low': 50105, 'Medium': 49997, 'High': 49898}
City: {'Singapore': 30236, 'New York': 30108, 'London': 29939, 'Mumbai': 29923, 'Sydney': 29794}


,Após ingestão,Após preparação
Registros,150000,150000
Atributos,21,21
Valores ausentes,0,0
Linhas duplicadas,0,0
Order_ID repetidos,0,0


**Resposta do grupo:**

A base já estava limpa, então a maior parte das decisões foi **manter os dados** (e o código de tratamento foi mantido por reprodutibilidade, removendo 0 registros):

- **D1 (ausentes), D2 (duplicados), D3 (tipos), D4 (valores impossíveis) e D5 (categorias):** nenhum problema encontrado; 0 registros removidos e nenhum valor alterado.
- **D6 (tempo × distância, Q9):** mantidos todos os registros. Tempo e distância são praticamente independentes (correlação de 0,004), o que é coerente com uma base sintética; remover as velocidades extremas seria arbitrário e poderia distorcer a pergunta P1. A limitação fica registrada.
- **D7 (lucro maior que a receita, Q13):** mantidos os 26.255 registros (17,5%). O Kaggle não documenta como `Revenue_USD` e `Profit_USD` foram calculados, e remover essa fração da base sem critério seguro seria arbitrário. A margem (lucro/receita) deve ser interpretada com cautela nas próximas etapas.

Resultado: `df_clean` com os mesmos 150.000 registros e 21 colunas, sem ausentes e sem duplicados.

## 5. Transformação dos dados

Realize as transformações necessárias à análise, como criação de colunas, filtros, agrupamentos, conversões ou junções, quando aplicáveis.

In [12]:
# Salva o conjunto preparado (nenhuma transformação analítica foi aplicada)
df_clean.to_csv("data/processed/food_delivery_clean.csv", index=False)
print("Salvo:", df_clean.shape)

Salvo: (150000, 21)


**Resposta do grupo:**

Nenhuma transformação analítica foi necessária neste RA (sem novas colunas, filtros, agrupamentos ou junções): o RA1 se limitou à ingestão, avaliação da qualidade e limpeza. O conjunto preparado foi apenas salvo em `data/processed/food_delivery_clean.csv`. Agregações e indicadores ficam para as próximas etapas.

## 7. Considerações do RA1

Registre brevemente:

- O que o grupo descobriu sobre os dados.
- Quais dificuldades ou limitações encontrou.
- Como os dados preparados poderão ser utilizados nas próximas etapas.

**Resposta do grupo:**

- **O que descobrimos:** a base é sintética, grande (150.000 pedidos) e muito limpa: sem ausentes, duplicados, tipos incorretos, valores impossíveis ou outliers pelo IQR.
- **Dificuldades e limitações:** (1) a base não tem ano nem data completa, só mês e trimestre; (2) tempo e distância quase não se relacionam (correlação de 0,004), com velocidades irreais em parte das entregas; (3) 17,5% dos pedidos têm lucro maior que a receita e o cálculo dessas colunas não é documentado; (4) por ser sintética, as relações entre variáveis podem não refletir a realidade.
- **Uso nas próximas etapas:** `df_clean` (150.000 × 21) está pronto como ponto de partida para transformação e análise, com as limitações acima registradas. No RA2, as faixas de valores validadas aqui orientam a geração dos registros simulados do streaming.